In [1]:
import os
# İşte sihirli değnek:
# Bu ortam değişkeni, PyTorch'un bazı işlemlerde bfloat16'ya geçmesini engeller.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    TrainerCallback
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

class GPUMonitorCallback(TrainerCallback):

    """Her log adımında Loss ve GPU durumunu ekrana basar."""
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_local_process_zero and logs is not None:
            step = state.global_step
            loss = logs.get("loss", "N/A")

            print(f"🔄 Adım: {step} | 📉 Kayıp (Loss): {loss}")

            if torch.cuda.is_available():
                allocated = torch.cuda.memory_allocated(0) / (1024**3)
                reserved = torch.cuda.memory_reserved(0) / (1024**3)
                print(f"💻 GPU Bellek -> Aktif: {allocated:.2f} GB | Rezerve: {reserved:.2f} GB")

            print("-" * 50)


# ============================================================
# 1. Basic configuration
# ============================================================

MODEL_NAME = r"../../NLP/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
DATASET_NAME = "Renicames/turkish-law-chatbot"

OUTPUT_DIR = "./qwen2_5_1_5b_turkish_law_qlora"
MAX_SEQ_LENGTH = 512   # safer for 8 GB VRAM. Try 1024 later if it fits.

SYSTEM_PROMPT = (
    "Sen Türk hukuku konusunda yardımcı olan bir asistansın. "
    "Sorulara kısa, açık ve hukuki doğruluğa dikkat ederek cevap ver."
)


# GPU'nun bfloat16'yı görmesini engelliyoruz:
os.environ["ACCELERATE_DISABLE_RICH"] = "1" 
os.environ["TORCH_CUDNN_V8_API_ENABLED"] = "1"




# ============================================================
# 2. Load dataset
# ============================================================

ds = load_dataset(DATASET_NAME)

print(ds)
print("Train columns:", ds["train"].column_names)
print("First sample:", ds["train"][0])

# If the dataset has no validation split, create one from train.
split = ds["train"].train_test_split(test_size=0.1, seed=42, shuffle=True)
train_ds = split["train"]
val_ds = split["test"]

# If the dataset has a test split, keep it for final evaluation only.
test_ds = ds["test"] if "test" in ds else None


# ============================================================
# 3. Detect dataset columns
# ============================================================

def detect_columns(sample):
    keys = sample.keys()

    possible_question_cols = ["Soru", "soru", "question", "instruction", "input", "prompt"]
    possible_answer_cols = ["Cevap", "cevap", "answer", "output", "response", "completion"]

    question_col = None
    answer_col = None

    for col in possible_question_cols:
        if col in keys:
            question_col = col
            break

    for col in possible_answer_cols:
        if col in keys:
            answer_col = col
            break

    if question_col is None or answer_col is None:
        raise ValueError(
            f"Could not detect question/answer columns. Available columns: {list(keys)}"
        )

    return question_col, answer_col


QUESTION_COL, ANSWER_COL = detect_columns(train_ds[0])

print("Detected question column:", QUESTION_COL)
print("Detected answer column:", ANSWER_COL)


# ============================================================
# 4. Load tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"


# ============================================================
# 5. Convert dataset into chat-template text
# ============================================================

def format_example(example):
    question = str(example[QUESTION_COL]).strip()
    answer = str(example[ANSWER_COL]).strip()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}


train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
val_ds = val_ds.map(format_example, remove_columns=val_ds.column_names)

print("Formatted sample:")
print(train_ds[0]["text"][:1000])


# ============================================================
# 6. 4-bit QLoRA model loading
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": "cuda:0"},
    dtype=torch.float16,
    trust_remote_code=True,
)

model.config.dtype = torch.float16

model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    )

# ============================================================
# 7. LoRA configuration
# ============================================================

# Qwen uses projection modules with these names.
lora_config = LoraConfig(
    r=8,                    # safer for 8 GB VRAM. Try 16 later if stable.
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


# ============================================================
# 8. Training arguments (Artık SFTConfig kullanıyoruz)
# ============================================================

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=2,

    max_length=MAX_SEQ_LENGTH,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    logging_steps=25,

    save_strategy="steps",
    save_steps=500,

    eval_strategy="steps",
    eval_steps=500,

    fp16=False,
    bf16=False,

    tf32=False,

    optim="adamw_torch",

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    report_to="none",

    save_total_limit=2,
)


# ============================================================
# 9. Trainer (Artık çok daha sade)
# ============================================================

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
    callbacks=[GPUMonitorCallback()]
)

# ============================================================
# 10. Train
# ============================================================


trainer.train()


# ============================================================
# 11. Save LoRA adapter
# ============================================================

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Training complete.")
print("Saved LoRA adapter to:", OUTPUT_DIR)

DatasetDict({
    train: Dataset({
        features: ['Soru', 'Cevap'],
        num_rows: 13354
    })
    test: Dataset({
        features: ['Soru', 'Cevap'],
        num_rows: 1500
    })
})
Train columns: ['Soru', 'Cevap']
First sample: {'Soru': "Anayasa madde 1'e göre, türkiye'nin devlet şekli nedir", 'Cevap': "Anayasa madde 1'e göre, türkiye'nin devlet şekli cumhuriyettir. bu madde, türkiye'nin yönetim biçiminin halkın egemenliğine dayandığını ve bu yönetim biçiminin cumhuriyet olduğunu belirler. cumhuriyet, halkın kendi kendini yönetme biçimi olarak kabul edilir ve türkiye cumhuriyeti'nin temel yönetim ilkesi olarak anayasal güvence altına alınmıştır."}
Detected question column: Soru
Detected answer column: Cevap
Formatted sample:
<|im_start|>system
Sen Türk hukuku konusunda yardımcı olan bir asistansın. Sorulara kısa, açık ve hukuki doğruluğa dikkat ederek cevap ver.<|im_end|>
<|im_start|>user
Hukuki işlemde irade sakatlığı nedir?<|im_end|>
<|im_start|>assistant
Hukuki işlemde i

W0520 01:41:42.192000 6696 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

C:\Users\cihat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
500,0.736098,0.732506
1000,0.628865,0.623105
1500,0.536039,0.559939
2000,0.501832,0.530989
2500,0.497015,0.520310
3000,0.455131,0.519783
3006,0.455131,0.519784


🔄 Adım: 25 | 📉 Kayıp (Loss): 2.8595657348632812
💻 GPU Bellek -> Aktif: 1.58 GB | Rezerve: 3.17 GB
--------------------------------------------------
🔄 Adım: 50 | 📉 Kayıp (Loss): 1.719812469482422
💻 GPU Bellek -> Aktif: 1.58 GB | Rezerve: 3.17 GB
--------------------------------------------------
🔄 Adım: 75 | 📉 Kayıp (Loss): 1.238886489868164
💻 GPU Bellek -> Aktif: 1.58 GB | Rezerve: 3.17 GB
--------------------------------------------------
🔄 Adım: 100 | 📉 Kayıp (Loss): 1.107513198852539
💻 GPU Bellek -> Aktif: 1.58 GB | Rezerve: 3.17 GB
--------------------------------------------------
🔄 Adım: 125 | 📉 Kayıp (Loss): 1.0602352905273438
💻 GPU Bellek -> Aktif: 1.58 GB | Rezerve: 3.17 GB
--------------------------------------------------
🔄 Adım: 150 | 📉 Kayıp (Loss): 0.9801789855957032
💻 GPU Bellek -> Aktif: 1.58 GB | Rezerve: 3.17 GB
--------------------------------------------------
🔄 Adım: 175 | 📉 Kayıp (Loss): 0.9799674224853515
💻 GPU Bellek -> Aktif: 1.58 GB | Rezerve: 3.17 GB
-------